# Dataset 4: patient, admission, laboratory, and demographic EDA

This split contains the patient identifier, admission type, laboratory-procedure count, medication statuses, and race.

The notebook analyses this split independently. The original CSV is not changed; the cleaned copy is saved in `EDA_Dataset` for later joining on `encounter_id`.

## Setup

In [1]:
from pathlib import Path  # Import required library

import matplotlib.pyplot as plt  # Import required library
import pandas as pd  # Import required library
import seaborn as sns  # Import required library

sns.set_theme(style="whitegrid", context="notebook")  # Set the chart theme
pd.set_option("display.max_rows", 100)  # Configure dataframe display
pd.set_option("display.max_columns", 50)  # Configure dataframe display

In [2]:
def show_category_counts(series, title, top_n=15):  # Define the categorical-frequency chart helper
    counts = series.fillna("Missing").value_counts().head(top_n).sort_values()  # Calculate category frequencies
    fig, ax = plt.subplots(figsize=(10, max(3.5, 0.38 * len(counts))))  # Create the chart area
    ax.barh(counts.index.astype(str), counts.values, color="#4C78A8")  # Draw the horizontal bar chart
    ax.set_title(f"{title}: top {top_n} categories")  # Label the chart axis or title
    ax.set_xlabel("Encounter count")  # Label the chart axis or title
    ax.set_ylabel("")  # Label the chart axis or title
    for y, value in enumerate(counts.values):  # Add value labels to each bar
        ax.text(value, y, f" {value:,}", va="center", fontsize=8)  # Run this EDA or data-cleaning step
    plt.tight_layout()  # Adjust chart spacing
    plt.show()  # Display the chart


def show_numeric_distribution(series, title, discrete=False):  # Define the numeric-distribution chart helper
    values = series.dropna()  # Remove missing numeric values before plotting
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))  # Create the chart area
    sns.histplot(values, discrete=discrete, ax=axes[0], color="#4C78A8")  # Plot the feature distribution
    axes[0].set_title(f"{title}: distribution")  # Configure the numeric chart
    axes[0].set_xlabel(title)  # Configure the numeric chart
    sns.boxplot(x=values, ax=axes[1], color="#72B7B2")  # Plot the feature box plot
    axes[1].set_title(f"{title}: box plot")  # Configure the numeric chart
    axes[1].set_xlabel(title)  # Configure the numeric chart
    plt.tight_layout()  # Adjust chart spacing
    plt.show()  # Display the chart

## Load the source data

In [3]:
source_path = Path("divided_datasets") / "dataset_4.csv"  # Set the path to the source CSV
df_raw = pd.read_csv(source_path, na_values=["?"], keep_default_na=True, low_memory=False)  # Load the source CSV and convert question marks to missing values

## Initial inspection

In [4]:
df_raw.head()  # Inspect the original dataframe

,encounter_id,patient_nbr,glimepiride-pioglitazone,pioglitazone,admission_type_id,glipizide,num_lab_procedures,troglitazone,race
0,2278392,8222157,No,No,6,No,41,No,Caucasian
1,149190,55629189,No,No,1,No,59,No,Caucasian
2,64410,86047875,No,No,1,Steady,11,No,AfricanAmerican
3,500364,82442376,No,No,1,No,44,No,Caucasian
4,16680,42519267,No,No,1,Steady,51,No,Caucasian


In [5]:
df_raw.info()  # Inspect the original dataframe

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 9 columns):
 #   Column                    Non-Null Count   Dtype
---  ------                    --------------   -----
 0   encounter_id              101766 non-null  int64
 1   patient_nbr               101766 non-null  int64
 2   glimepiride-pioglitazone  101766 non-null  str  
 3   pioglitazone              101766 non-null  str  
 4   admission_type_id         101766 non-null  int64
 5   glipizide                 101766 non-null  str  
 6   num_lab_procedures        101766 non-null  int64
 7   troglitazone              101766 non-null  str  
 8   race                      99493 non-null   str  
dtypes: int64(4), str(5)
memory usage: 8.8 MB


In [6]:
df_raw.shape  # Inspect the original dataframe

(101766, 9)

In [7]:
df_raw.describe()  # Inspect the original dataframe

,encounter_id,patient_nbr,admission_type_id,num_lab_procedures
count,1.017660e+05,1.017660e+05,101766.000000,101766.000000
mean,1.652016e+08,5.433040e+07,2.024006,43.095641
std,1.026403e+08,3.869636e+07,1.445403,19.674362
min,1.252200e+04,1.350000e+02,1.000000,1.000000
25%,8.496119e+07,2.341322e+07,1.000000,31.000000
50%,1.523890e+08,4.550514e+07,1.000000,44.000000
75%,2.302709e+08,8.754595e+07,3.000000,57.000000
max,4.438672e+08,1.895026e+08,8.000000,132.000000


In [8]:
df_raw.isna().sum().sort_values(ascending=False).to_frame(name="missing_count")  # Inspect the original dataframe

,missing_count
race,2273
encounter_id,0
patient_nbr,0
pioglitazone,0
glimepiride-pioglitazone,0
admission_type_id,0
glipizide,0
num_lab_procedures,0
troglitazone,0


In [9]:
df_raw.duplicated().sum()  # Inspect the original dataframe

np.int64(0)

In [10]:
df_raw["encounter_id"].duplicated().sum()  # Run this EDA or data-cleaning step

np.int64(0)

In [11]:
df_raw["encounter_id"].is_unique  # Run this EDA or data-cleaning step

True

In [12]:
df_raw.nunique(dropna=False).sort_values().to_frame(name="unique_values_including_missing")  # Inspect the original dataframe

,unique_values_including_missing
glimepiride-pioglitazone,2
troglitazone,2
glipizide,4
pioglitazone,4
race,6
admission_type_id,8
num_lab_procedures,118
patient_nbr,71518
encounter_id,101766


## Cleaning decisions

- Missing-value normalization: Question-mark and blank text values are treated as missing.
- Race: A missingness flag is retained and missing race values are labeled Unknown for visualization.
- Identifiers: patient_nbr is preserved because repeat patient encounters can be useful in later analysis. encounter_id remains the join key.
- Rows and join key: No rows are removed. encounter_id remains unchanged and unique for the later join.

In [13]:
df = df_raw.copy()  # Copy the source dataframe before cleaning

text_columns = df.select_dtypes(include=["object", "string"]).columns  # Identify text columns for normalization
for column in text_columns:  # Clean each text column
    df[column] = df[column].astype("string").str.strip().replace("", pd.NA)  # Strip whitespace and convert blank text to missing values

df["race_missing"] = df["race"].isna()  # Create or handle race missingness information
df["race"] = df["race"].fillna("Unknown")  # Create or handle race missingness information
columns_removed = []  # Record columns removed during cleaning

assert len(df) == len(df_raw), "No encounter rows should be removed"  # Validate the cleaning result
assert df["encounter_id"].notna().all(), "The join key must not be missing"  # Validate the cleaning result
assert df["encounter_id"].is_unique, "The join key must remain unique"  # Validate the cleaning result
assert df.columns.is_unique, "Duplicate column names are not allowed"  # Validate the cleaning result

In [14]:
cleaning_summary = pd.DataFrame({  # Create the cleaning-decision summary
    "decision": ["Missing-value normalization","Race","Identifiers","Rows and join key","Columns removed"],  # Add a cleaning-summary field
    "detail": ["Question-mark and blank text values are treated as missing.","A missingness flag is retained and missing race values are labeled Unknown for visualization.","patient_nbr is preserved because repeat patient encounters can be useful in later analysis. encounter_id remains the join key.","No rows are removed. encounter_id remains unchanged and unique for the later join.","Columns removed after review: none"]  # Add a cleaning-summary field
})  # Run this EDA or data-cleaning step
cleaning_summary  # Create the cleaning-decision summary

,decision,detail
0,Missing-value normalization,Question-mark and blank text values are treate...
1,Race,A missingness flag is retained and missing rac...
2,Identifiers,patient_nbr is preserved because repeat patien...
3,Rows and join key,No rows are removed. encounter_id remains unch...
4,Columns removed,Columns removed after review: none


## Post-cleaning validation

In [15]:
df.head()  # Run this EDA or data-cleaning step

,encounter_id,patient_nbr,glimepiride-pioglitazone,pioglitazone,admission_type_id,glipizide,num_lab_procedures,troglitazone,race,race_missing
0,2278392,8222157,No,No,6,No,41,No,Caucasian,False
1,149190,55629189,No,No,1,No,59,No,Caucasian,False
2,64410,86047875,No,No,1,Steady,11,No,AfricanAmerican,False
3,500364,82442376,No,No,1,No,44,No,Caucasian,False
4,16680,42519267,No,No,1,Steady,51,No,Caucasian,False


In [16]:
df.info()  # Run this EDA or data-cleaning step

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 10 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   glimepiride-pioglitazone  101766 non-null  string
 3   pioglitazone              101766 non-null  string
 4   admission_type_id         101766 non-null  int64 
 5   glipizide                 101766 non-null  string
 6   num_lab_procedures        101766 non-null  int64 
 7   troglitazone              101766 non-null  string
 8   race                      101766 non-null  string
 9   race_missing              101766 non-null  bool  
dtypes: bool(1), int64(4), string(5)
memory usage: 8.9 MB


In [17]:
df.shape  # Run this EDA or data-cleaning step

(101766, 10)

In [18]:
df.describe()  # Run this EDA or data-cleaning step

,encounter_id,patient_nbr,admission_type_id,num_lab_procedures
count,1.017660e+05,1.017660e+05,101766.000000,101766.000000
mean,1.652016e+08,5.433040e+07,2.024006,43.095641
std,1.026403e+08,3.869636e+07,1.445403,19.674362
min,1.252200e+04,1.350000e+02,1.000000,1.000000
25%,8.496119e+07,2.341322e+07,1.000000,31.000000
50%,1.523890e+08,4.550514e+07,1.000000,44.000000
75%,2.302709e+08,8.754595e+07,3.000000,57.000000
max,4.438672e+08,1.895026e+08,8.000000,132.000000


In [19]:
df.isna().sum().sort_values(ascending=False).to_frame(name="missing_count_after_cleaning")  # Run this EDA or data-cleaning step

,missing_count_after_cleaning
encounter_id,0
patient_nbr,0
glimepiride-pioglitazone,0
pioglitazone,0
admission_type_id,0
glipizide,0
num_lab_procedures,0
troglitazone,0
race,0
race_missing,0


In [20]:
df["encounter_id"].nunique()  # Inspect the selected cleaned column

101766

## Identifier check

### Patient identifier coverage

In [21]:
df["patient_nbr"].nunique()  # Inspect the selected cleaned column

71518

## Univariate EDA after cleaning

### Number of laboratory procedures

In [22]:
df["num_lab_procedures"].describe()  # Inspect the selected cleaned column

count    101766.000000
mean         43.095641
std          19.674362
min           1.000000
25%          31.000000
50%          44.000000
75%          57.000000
max         132.000000
Name: num_lab_procedures, dtype: float64

In [23]:
show_numeric_distribution(df["num_lab_procedures"], "Number of laboratory procedures", discrete=True)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_3632\4155684497.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Glimepiride-pioglitazone status

In [24]:
df["glimepiride-pioglitazone"].value_counts(dropna=False).head(10).to_frame(name="count")  # Inspect the selected cleaned column

,count
glimepiride-pioglitazone,
No,101765
Steady,1


In [25]:
show_category_counts(df["glimepiride-pioglitazone"], "Glimepiride-pioglitazone status", top_n=10)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_3632\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Pioglitazone status

In [26]:
df["pioglitazone"].value_counts(dropna=False).head(10).to_frame(name="count")  # Inspect the selected cleaned column

,count
pioglitazone,
No,94438
Steady,6976
Up,234
Down,118


In [27]:
show_category_counts(df["pioglitazone"], "Pioglitazone status", top_n=10)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_3632\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Admission type ID

In [28]:
df["admission_type_id"].value_counts(dropna=False).head(20).to_frame(name="count")  # Inspect the selected cleaned column

,count
admission_type_id,
1,53990
3,18869
2,18480
6,5291
5,4785
8,320
7,21
4,10


In [29]:
show_category_counts(df["admission_type_id"], "Admission type ID", top_n=20)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_3632\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Glipizide status

In [30]:
df["glipizide"].value_counts(dropna=False).head(10).to_frame(name="count")  # Inspect the selected cleaned column

,count
glipizide,
No,89080
Steady,11356
Up,770
Down,560


In [31]:
show_category_counts(df["glipizide"], "Glipizide status", top_n=10)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_3632\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Troglitazone status

In [32]:
df["troglitazone"].value_counts(dropna=False).head(10).to_frame(name="count")  # Inspect the selected cleaned column

,count
troglitazone,
No,101763
Steady,3


In [33]:
show_category_counts(df["troglitazone"], "Troglitazone status", top_n=10)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_3632\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Race

In [34]:
df["race"].value_counts(dropna=False).head(10).to_frame(name="count")  # Create or handle race missingness information

,count
race,
Caucasian,76099
AfricanAmerican,19210
Unknown,2273
Hispanic,2037
Other,1506
Asian,641


In [35]:
show_category_counts(df["race"], "Race", top_n=10)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_3632\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


## Save the cleaned split

In [36]:
output_dir = Path("EDA_Dataset")  # Set the cleaned-data output folder
output_dir.mkdir(exist_ok=True)  # Set the cleaned-data output folder
output_path = output_dir / "dataset_4.csv"  # Set the cleaned CSV output path
df.to_csv(output_path, index=False)  # Save the cleaned dataframe
print(f"Saved {df.shape[0]:,} rows and {df.shape[1]:,} columns to {output_path}")  # Report the saved dataset details

Saved 101,766 rows and 10 columns to EDA_Dataset\dataset_4.csv


In [37]:
saved_df = pd.read_csv(output_path)  # Reload the saved CSV for verification
assert saved_df.shape == df.shape, "Saved CSV shape does not match the cleaned dataframe"  # Validate the cleaning result
saved_df.head()  # Reload the saved CSV for verification

,encounter_id,patient_nbr,glimepiride-pioglitazone,pioglitazone,admission_type_id,glipizide,num_lab_procedures,troglitazone,race,race_missing
0,2278392,8222157,No,No,6,No,41,No,Caucasian,False
1,149190,55629189,No,No,1,No,59,No,Caucasian,False
2,64410,86047875,No,No,1,Steady,11,No,AfricanAmerican,False
3,500364,82442376,No,No,1,No,44,No,Caucasian,False
4,16680,42519267,No,No,1,Steady,51,No,Caucasian,False
